# SETUP

In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import time

In [2]:
print("JAX version:", jax.__version__)
print("JAX XLA backend platform:", jax.lib.xla_bridge.get_backend().platform)
print("Available devices:", jax.devices())

JAX version: 0.4.30
JAX XLA backend platform: gpu
Available devices: [cuda(id=0)]


In [3]:
x = jnp.ones((1000, 1000))
y = jnp.dot(x, x)
print("Computation device:", x.devices())
print("Computation device:", y.devices())

x_np = np.ones((1000, 1000))
y_np = np.dot(x, x)
x_jnp = jnp.array(x_np)
y_jnp = jnp.dot(x_np, x_np)
print("Computation device:", x_jnp.devices())
print("Computation device:", y_jnp.devices())

Computation device: {cuda(id=0)}
Computation device: {cuda(id=0)}
Computation device: {cuda(id=0)}
Computation device: {cuda(id=0)}


# JIT

In [4]:
# input tensors
A_np = np.random.rand(100, 100)
B_np = np.random.rand(100, 100)

A_jax = jnp.array(A_np)
B_jax = jnp.array(B_np)

In [5]:
# function for native JIT
def jax_function(A, B):
	C = B
	for _ in range(50):
		C = A @ C
		C = (C + A + B) / 3
		C = C * A * B
		C = jnp.transpose(C)
		C = jnp.linalg.pinv(C)
	d = jnp.linalg.det(C)
	return d

# trace & warm up
jax_function_jit = jax.jit(jax_function)
for _ in range(100): yt = jax_function_jit(A_jax, B_jax).block_until_ready()

In [ ]:
# no JIT

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function(A_jax, B_jax).block_until_ready()
	end = time.time()
	runtimes.append(end - start)
print(f"raw avg time: {sum(runtimes)/len(runtimes):.6f} seconds")

JAX avg time: 0.197356 seconds


In [ ]:
# JIT

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function_jit(A_jax, B_jax).block_until_ready()
	end = time.time()
	runtimes.append(end - start)
print(f"JIT avg time: {sum(runtimes)/len(runtimes):.6f} seconds")

JAX avg time (JIT): 0.154284 seconds


# AUTOGRAD

In [8]:
# grad, trace & warm up
jax_function_grad = jax.grad(jax_function)

jax_function_grad_jit = jax.jit(jax_function_grad)
for _ in range(100): yt = jax_function_grad_jit(A_jax, B_jax).block_until_ready()

In [9]:
# no JIT

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function_grad(A_jax, B_jax).block_until_ready()
	end = time.time()
	runtimes.append(end - start)
print(f"raw avg time: {sum(runtimes)/len(runtimes):.6f} seconds")

JAX avg time: 1.178381 seconds


In [10]:
# JIT

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function_grad_jit(A_jax, B_jax).block_until_ready()
	end = time.time()
	runtimes.append(end - start)
print(f"JIT avg time: {sum(runtimes)/len(runtimes):.6f} seconds")

JIT avg time: 0.160851 seconds
